In [5]:
using ComputableDAGs
using QEDFeynmanDiagrams
using RuntimeGeneratedFunctions
using Plots
using DataFrames
using BenchmarkTools
using QEDcore, QEDprocesses
using Logging

RuntimeGeneratedFunctions.init(@__MODULE__)

global_logger(NullLogger())

Base.CoreLogging.NullLogger()

In [6]:
function profile(instance, input, optimizer, closures_size)
    b_gen = @benchmark graph($instance)
    g = graph(instance)

    if !isnothing(optimizer)
        # compile run first
        begin
            g_temp = graph(instance)
            optimize_to_fixpoint!(optimizer, g_temp)
        end

        t_optimization = @elapsed optimize_to_fixpoint!(optimizer, g)
    else
        b_optimization = [0.]
    end

    g_props = get_properties(g)

    b_fgen = @benchmark get_compute_function($g, $instance, cpu_st(), @__MODULE__; closures_size=$closures_size)
    f = get_compute_function(g, instance, cpu_st(), @__MODULE__; closures_size=closures_size)

    println(f)

    tic = time_ns()
    f(input)
    toc = time_ns()
    t_compile = toc - tic

    b_exec = @benchmark $f($input)
    
    return (
        instance=string(instance),
        optimizer=optimizer,
        closures_size=closures_size,
        b_gen=b_gen,
        b_optimization=b_optimization,
        g_props=g_props,
        b_fgen=b_fgen,
        t_compile=t_compile,
        b_exec=b_exec,
    )
end

profile (generic function with 1 method)

In [7]:
function make_nphoton_compton(n::Int, all_combs::Bool)
    return ScatteringProcess(
        (Electron(), ntuple(_ -> Photon(), n)...),     # incoming particles
        (Electron(), Photon()),                        # outgoing particles
        (all_combs ? AllSpin() : SpinUp(), ntuple(_ -> all_combs ? AllPol() : PolX(), n)...),  # incoming particle spin/pols
        (all_combs ? AllSpin() : SpinUp(), all_combs ? AllPol() : PolX()),                         # outgoing particle spin/pols
    )
end

make_nphoton_compton (generic function with 1 method)

In [8]:
df = DataFrame()

MODEL = PerturbativeQED()

SCATTERING_PROCESSES = [
    make_nphoton_compton(1, false),
    #=make_nphoton_compton(2, false),
    make_nphoton_compton(3, false),
    make_nphoton_compton(4, false),=#
]
for (INSTANCE, OPTIMIZER, CLOSURE_SIZE) in Iterators.product(SCATTERING_PROCESSES, (nothing, ReductionOptimizer(), SplitOptimizer()), (0, 100))
    psp = PhaseSpacePoint(
        INSTANCE,
        MODEL,
        PhasespaceDefinition(SphericalCoordinateSystem(), ElectronRestFrame()),
        tuple((rand(SFourMomentum) for _ in 1:number_incoming_particles(INSTANCE))...),
        tuple((rand(SFourMomentum) for _ in 1:number_outgoing_particles(INSTANCE))...),
    )
    println("$INSTANCE | $OPTIMIZER | $CLOSURE_SIZE")
    results = profile(INSTANCE, psp, OPTIMIZER, CLOSURE_SIZE)
    push!(df,
        results
    )
end

df

generic QED process "ek -> ek" | nothing | 0
RuntimeGeneratedFunction{(:input,), ComputableDAGs.var"#_RGF_ModTag", var"#_RGF_ModTag", (0xdbcd0945, 0x4bbb79ca, 0xb4408bf6, 0x2d9da147, 0x8a8038ce), Expr}(quote
    begin
        _9d56d0d4_ac05_11ef_2b9b_bfdce35c1da5_in = (var"#59368#59369"())(input)
        _9d56d0c0_ac05_11ef_2466_13c1c29f6eb2_in = (var"#59370#59371"())(input)
        _9d56d0c8_ac05_11ef_0d71_cdc555a07143_in = (var"#59372#59373"())(input)
        _9d56d0de_ac05_11ef_26d9_33c162a9ad2e_in = (var"#59374#59375"())(input)
        _9d56d0f2_ac05_11ef_1f7e_418bde4b2cb7_in = (var"#59376#59377"())(input)
        _9d56d0e8_ac05_11ef_285d_1dbb7f2bf283_in = (var"#59378#59379"())(input)
    end
    begin
        _9d56d0d4_ac05_11ef_2b9b_bfdce35c1da5 = (identity)(_9d56d0d4_ac05_11ef_2b9b_bfdce35c1da5_in)
        _9d56d0d4_ac05_11ef_125a_9315e1682ffa = (ComputableDAGs.compute)(QEDFeynmanDiagrams.ComputeTask_BaseState(), _9d56d0d4_ac05_11ef_2b9b_bfdce35c1da5)
        _9d56d0d4_ac05_11ef

MethodError: MethodError: no method matching (::var"#59368#59369")(::PhaseSpacePoint{ScatteringProcess{Tuple{Electron, Photon}, Tuple{Electron, Photon}, Tuple{SpinUp, PolarizationX}, Tuple{SpinUp, PolarizationX}}, PerturbativeQED, PhasespaceDefinition{SphericalCoordinateSystem, ElectronRestFrame}, Tuple{ParticleStateful{Incoming, Electron, SFourMomentum}, ParticleStateful{Incoming, Photon, SFourMomentum}}, Tuple{ParticleStateful{Outgoing, Electron, SFourMomentum}, ParticleStateful{Outgoing, Photon, SFourMomentum}}, SFourMomentum})
The applicable method may be too new: running in world age 46697, while current world is 61335.

Closest candidates are:
  (::var"#59368#59369")(::Any) (method too new to be called from this world context.)
   @ Main none:0
